# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the [FAIR<sup>2</sup>](https://sen.science/doi/10.71728/senscience.vq4a-28xa) mass spectrometry dataset using the `mlcroissant` library, referencing all records, fields, and columns by their unique `@id` as per the Croissant schema.

### Dataset Source
The dataset is described by a Croissant schema available at:
- https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets, fields, and columns by their @id
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set['@id']}, name: {record_set.get('name', 'N/A')}")
    if 'fields' in record_set and record_set['fields']:
        print("  Fields and their columns:")
        for field in record_set['fields']:
            print(f"    - Field @id: {field['@id']}, name: {field.get('name', 'N/A')}, columns: {[col['@id'] for col in field.get('columns', [])]}")
    else:
        print("  [No fields described in this record set]")

Let's look at an example of record iteration for one of the record sets. Below, records are listed using their `@id`.

In [ ]:
# List records from a selected record set by its @id.
# Here we select the first available record set for demonstration.
record_sets = [record_set['@id'] for record_set in metadata.record_sets]
if record_sets:
    chosen_record_set_id = record_sets[0]
    print(f"Sample records for RecordSet @id: {chosen_record_set_id}")
    for idx, record in enumerate(dataset.records(record_set=chosen_record_set_id)):
        print(record)
        if idx >= 2:
            break
else:
    print("No record sets were found in this dataset.")

## 3. Data Extraction
Load data from specific record sets into a DataFrame for analysis. All processing references record set and field/column `@id`s listed earlier.

In [ ]:
# Extract data for each record set by @id
dataframes = {}

for rs_id in record_sets:
    records = list(dataset.records(record_set=rs_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"\nRecordSet @id: {rs_id}, shape: {df.shape}")
        print(f"Fields: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for RecordSet @id: {rs_id}")

# Use the first available record set for deeper processing
if record_sets:
    first_rs = record_sets[0]
    df = dataframes[first_rs] if first_rs in dataframes else None
else:
    first_rs = None
    df = None

## 4. Exploratory Data Analysis (EDA)
We'll demonstrate typical EDA steps: filtering, normalizing, and grouping data by relevant fields (using the field and column `@id` from Croissant schema).

Adjust the values below to target different fields or use additional ones as needed for your analysis.

In [ ]:
import numpy as np

# Example: Select a numeric field by its @id (update based on your overview above)
if df is not None and len(df.columns) > 0:
    # Heuristically select a numeric-looking column using pandas dtype inference
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        numeric_field_id = numeric_columns[0]  # First numeric field (by @id)
    else:
        numeric_field_id = df.columns[0]  # fallback
    print(f"Selected numeric field @id: {numeric_field_id}")

    # Filter for values above threshold
    threshold = df[numeric_field_id].mean() if numeric_field_id in df else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold} (mean):")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean())
        / filtered_df[numeric_field_id].std()
    )
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (if present, e.g. the second column or one with low cardinality)
    group_field_id = None
    candidate_categorical = [col for col in df.columns if col != numeric_field_id]
    for col in candidate_categorical:
        if df[col].nunique() < 20:
            group_field_id = col
            break
    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
        print(f"Grouped data by {group_field_id}:")
        display(grouped_df.head())
    else:
        print("No suitable categorical group field found (with < 20 unique values)")
else:
    print("No suitable dataframe for EDA found.")

## 5. Visualization
Visualize distributions or relationships between fields in the dataset. Adjust field `@id`s as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if df is not None and numeric_field_id in df:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field_id].dropna(), bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    # If grouping field is available, boxplot
    if group_field_id and group_field_id in df:
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR<sup>2</sup> mass spectrometry analysis dataset. We reviewed schema elements by `@id`, demonstrated EDA processes, and visualized distributions for selected numeric and grouping fields. 

Key next steps include further cleaning, feature engineering, and domain-specific modeling/analysis. For deeper biological or experimental insights, consult schema metadata for field and record provenance or trace transformations by their schema `@id`s. 

Feel free to adjust field and record set `@id`s as new versions of the dataset are released or new schemas provided.